# Task Granularity, Stage Barriers and Speculative Execution

Companion notebook for the [DAG Scheduling, Stages and Tasks](https://codemia.io/courses/big_data_engines/bd_dag_scheduling_stages_and_tasks) lesson on Codemia.

**What you will build**: an event driven scheduler simulator: a fixed amount of work, a pool of slots where a few nodes are
slow, and a per task launch cost. You will find the task count that minimises wall clock time,
measure how much of a four stage job is spent waiting at barriers, price speculative execution in
both directions, and watch the speedup from more slots flatten out.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import heapq

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

SLOTS = 200             # task slots in the cluster
TOTAL_WORK_S = 24_000   # core-seconds of real computation in one stage
LAUNCH_S = 0.35         # per task: scheduling, serialisation, JVM warmth, first read
SLOW_SLOTS = 6          # a bad disk, a noisy neighbour, a throttled instance
SLOW_SPEED = 0.25       # those slots run at a quarter speed

speed = np.ones(SLOTS)
speed[rng.choice(SLOTS, SLOW_SLOTS, replace=False)] = SLOW_SPEED

print(f'{SLOTS} slots, {SLOW_SLOTS} of them at {SLOW_SPEED:.0%} speed')
print(f'effective capacity    {speed.sum():.1f} slots')
print(f'perfect packing floor {TOTAL_WORK_S / speed.sum():.1f} s for {TOTAL_WORK_S:,} core-seconds')

## Concept recap

A DAG engine turns a job into stages and a stage into tasks, one per partition. Inside a stage the
tasks are independent and the scheduler hands the next one to whichever slot is free. Between stages
there is a barrier: every task of a stage must finish before the next stage starts, because the
shuffle output a downstream task reads does not exist until its producers have written it.

Those two facts set the performance envelope of the whole job. Inside a stage you are running a
greedy packing problem whose makespan cannot be shorter than the longest single task. At every
barrier you pay for the slowest task in the stage while the rest of the cluster sits idle.

The cluster in this notebook has six slow slots in two hundred, which is the normal state of a real
cluster rather than a pathology. Everything below falls out of that one detail.

## An event driven scheduler

In [ ]:
def simulate(work, speed, speculate=False, start_frac=0.75, multiplier=1.5):
    """Run one stage. Tasks queue onto slots; a finished slot takes the next task.

    With speculate=True the scheduler behaves like a real engine: once start_frac of
    the tasks are done and a slot is free, it relaunches a task that has been running
    longer than multiplier times the median, and keeps whichever copy lands first.
    """
    n, slots = len(work), len(speed)
    avail = [(0.0, s) for s in range(slots)]
    heapq.heapify(avail)
    running, queue = [], list(range(n))
    done = np.zeros(n, bool)
    done_at = np.zeros(n)
    cloned = set()
    busy = extra = 0.0
    now = 0.0
    median = float(np.median(work))

    while queue or running:
        while queue and avail and avail[0][0] <= now:
            t_free, s = heapq.heappop(avail)
            task = queue.pop(0)
            start = max(t_free, now)
            d = work[task] / speed[s] + LAUNCH_S
            busy += d
            heapq.heappush(running, (start + d, task, s, start))
        if not running:
            now = avail[0][0]
            continue
        end, task, s, start = heapq.heappop(running)
        now = end
        heapq.heappush(avail, (now, s))
        if not done[task]:
            done[task], done_at[task] = True, now

        if speculate and not queue and done.sum() >= start_frac * n:
            for _, t2, _, st2 in sorted(running):
                if done[t2] or t2 in cloned or now - st2 < multiplier * median:
                    continue
                if not avail or avail[0][0] > now:
                    break
                _, s3 = heapq.heappop(avail)
                d = work[t2] / speed[s3] + LAUNCH_S
                busy += d
                extra += d
                cloned.add(t2)
                heapq.heappush(running, (now + d, t2, s3, now))
                break

    makespan = float(done_at.max())
    return {'makespan': makespan, 'busy': busy, 'extra': extra,
            'clones': len(cloned), 'slowest_task': float(done_at.max() - 0.0),
            'utilisation': busy / (makespan * slots),
            'idle': makespan * slots - busy}


def stage_work(n_tasks, total=TOTAL_WORK_S, sigma=0.35):
    """Equal partitions in theory, mildly unequal in practice. Total work is fixed."""
    w = rng.lognormal(0, sigma, n_tasks)
    return w * (total / w.sum())


COUNTS = [200, 400, 600, 1_000, 2_000, 4_000, 8_000, 20_000, 60_000, 150_000]
DRAWS = {n: stage_work(n) for n in COUNTS}      # drawn once, reused by every cell below
FLOOR = TOTAL_WORK_S / speed.sum()

for n in [200, 2_000]:
    r = simulate(DRAWS[n], speed)
    print(f'{n:>6,} tasks of ~{TOTAL_WORK_S / n:>6.1f} s: '
          f'makespan {r["makespan"]:>6.1f} s, utilisation {r["utilisation"]:>6.1%}, '
          f'{r["makespan"] / FLOOR:>5.2f}x the floor')

Two hundred tasks on two hundred slots is the configuration that feels obviously right and is
obviously wrong. Every slot gets exactly one task, so the six slow slots each take four times as long
as everyone else, and the stage ends when they do. Nothing can rebalance, because there is no work
left to hand to the slots that finished early.

Two thousand smaller tasks fix it without any change to the cluster. A slow slot becomes free later,
so the scheduler simply gives it fewer tasks: the greedy loop load balances automatically as long as
there is queued work left to balance with. That is the real argument for over partitioning, and it
has a limit on the other side, which the next cell finds.

## The granularity curve

In [ ]:
rows = []
for n in COUNTS:
    r = simulate(DRAWS[n], speed)
    rows.append((n, r, n * LAUNCH_S))

print(f'{"tasks":>9}{"work/task":>12}{"makespan":>11}{"vs floor":>10}'
      f'{"utilisation":>13}{"launch cost":>13}{"of real work":>14}')
for n, r, overhead in rows:
    print(f'{n:>9,}{TOTAL_WORK_S / n:>10.2f} s{r["makespan"]:>9.1f} s'
          f'{r["makespan"] / FLOOR:>9.2f}x{r["utilisation"]:>13.1%}'
          f'{overhead:>11,.0f} s{overhead / TOTAL_WORK_S:>13.1%}')

best = min(rows, key=lambda row: row[1]['makespan'])
print()
print(f'floor                   {FLOOR:>7.1f} s')
print(f'cheapest configuration  {best[1]["makespan"]:>7.1f} s at {best[0]:,} tasks '
      f'({best[0] / SLOTS:.0f} tasks per slot)')
print(f'default-ish 200 tasks   {rows[0][1]["makespan"]:>7.1f} s '
      f'({rows[0][1]["makespan"] / best[1]["makespan"]:.1f}x the cheapest)')

In [ ]:
ns = np.array([n for n, _, _ in rows])
makespans = np.array([r['makespan'] for _, r, _ in rows])
launch = np.array([o for _, _, o in rows]) / speed.sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.6))
ax1.plot(ns, makespans, 'o-', color='tab:blue', label='measured makespan')
ax1.axhline(FLOOR, color='black', linestyle='--', label='perfect packing floor')
ax1.plot(ns, FLOOR + launch, ':', color='tab:red', label='floor + launch cost')
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('tasks in the stage')
ax1.set_ylabel('stage wall clock (s)')
ax1.set_title('Left arm: slow slots. Right arm: launch cost.')
ax1.legend()
ax1.grid(alpha=0.3, which='both')

for n, colour in [(200, 'tab:red'), (2_000, 'tab:blue')]:
    w = DRAWS[n]
    avail = [(0.0, s) for s in range(SLOTS)]
    heapq.heapify(avail)
    queue, now = list(range(n)), 0.0
    last = np.zeros(SLOTS)
    running = []
    while queue or running:
        while queue and avail and avail[0][0] <= now:
            t_free, s = heapq.heappop(avail)
            task = queue.pop(0)
            start = max(t_free, now)
            end = start + w[task] / speed[s] + LAUNCH_S
            last[s] = end
            heapq.heappush(running, (end, s))
        if not running:
            now = avail[0][0]
            continue
        end, s = heapq.heappop(running)
        now = end
        heapq.heappush(avail, (now, s))
    ax2.hist(last, bins=40, alpha=0.6, color=colour, label=f'{n:,} tasks')
ax2.set_xlabel('when each slot went idle (s)')
ax2.set_ylabel('slots')
ax2.set_title('A stage ends when the last slot does')
ax2.set_xscale('log')
ax2.legend()
ax2.grid(alpha=0.3)
plt.show()

The right panel is the mechanism behind the left one. At one task per slot the idle times split into
two clumps, the fast slots and the slow ones, and the barrier waits for the far clump. With ten tasks
per slot every slot finishes at nearly the same moment, because the scheduler kept handing work to
whoever was ready and the slow slots simply received less of it.

The left arm of that curve is also visibly noisy, and the noise is the finding rather than an
artefact: at two hundred tasks the stage time depends on which particular task happened to land on a
slow slot, so the same job can take four hundred seconds one day and seven hundred the next with no
change anywhere. Over partitioning buys predictability as much as it buys speed.

So the sizing rule is not a magic ratio. It is: enough queued tasks that the greedy loop can
compensate for whatever your cluster is doing to you, and not so many that per task launch cost shows
up in the total. The usual advice of two to four tasks per slot is the left edge of that range, and
this curve says it is worth going further when your task launch cost is small.

## What a barrier costs

In [ ]:
STAGE_WORK = [24_000, 9_000, 9_000, 1_200]      # scan, shuffle aggregate, join, write
STAGE_TASKS = [2_000, 1_000, 1_000, 60]
LABELS = ['scan', 'aggregate', 'join', 'write']

print(f'{"stage":>11}{"tasks":>8}{"work":>11}{"makespan":>11}'
      f'{"utilisation":>13}{"idle slot-seconds":>20}')
total_makespan = total_busy = 0.0
results = {}
for label, work, n in zip(LABELS, STAGE_WORK, STAGE_TASKS):
    results[label] = r = simulate(stage_work(n, total=work), speed)
    total_makespan += r['makespan']
    total_busy += r['busy']
    print(f'{label:>11}{n:>8,}{work:>9,} s{r["makespan"]:>9.1f} s'
          f'{r["utilisation"]:>13.1%}{r["idle"]:>18,.0f} s')

print()
print(f'job wall clock                {total_makespan:>10.1f} s')
print(f'cluster seconds paid for      {total_makespan * SLOTS:>10,.0f}')
print(f'cluster seconds of real work  {total_busy:>10,.0f}')
print(f'utilisation across the job    {total_busy / (total_makespan * SLOTS):>10.1%}')
print()
fixed = simulate(stage_work(2_000, total=STAGE_WORK[-1]), speed)
write = results['write']
print(f'repartition before the write, 2,000 tasks instead of 60:')
print(f'  {write["makespan"]:.1f} s -> {fixed["makespan"]:.1f} s, '
      f'utilisation {write["utilisation"]:.1%} -> {fixed["utilisation"]:.1%}, '
      f'and {write["makespan"] - fixed["makespan"]:.0f} s off the job')

The write stage is where the arithmetic hurts. Sixty tasks on a two hundred slot cluster run at
sixteen percent utilisation, which means five sixths of the cluster is billed for doing nothing, and
it is idle because of a partition count inherited from whatever produced the final shuffle rather
than because of anything about writing files. Repartitioning before the write fixes it outright, and
the last line measures the fix.

The broader point is that a job's shape matters more than any single stage's speed. Four barriers
means paying four times for a slow tail, and a small partition count late in the plan, which looks
like tidiness, turns the cluster into a near single threaded writer.

## Speculative execution, priced in both directions

In [ ]:
print('a slow node is the case speculation was built for:')
print(f'{"tasks":>8}{"plain":>11}{"speculative":>14}{"saved":>9}'
      f'{"clones":>9}{"extra cluster time":>21}{"of real work":>14}')
for n in [200, 400, 1_000, 4_000]:
    w = DRAWS[n]
    plain = simulate(w, speed)
    spec = simulate(w, speed, speculate=True)
    print(f'{n:>8,}{plain["makespan"]:>9.1f} s{spec["makespan"]:>12.1f} s'
          f'{1 - spec["makespan"] / plain["makespan"]:>9.0%}{spec["clones"]:>9}'
          f'{spec["extra"]:>19,.0f} s{spec["extra"] / TOTAL_WORK_S:>14.1%}')

print()
print('now a healthy cluster, every slot at full speed, with one skewed partition.')
print('the only slow task is slow because of its data:')
healthy = np.ones(SLOTS)
print(f'{"tasks":>8}{"plain":>11}{"speculative":>14}{"saved":>9}'
      f'{"clones":>9}{"extra cluster time":>21}{"of real work":>14}')
for n in [400, 1_000]:
    w = DRAWS[n].copy()
    w[0] = 3 * FLOOR                                       # one partition, three floors of work
    w[1:] *= (TOTAL_WORK_S - w[0]) / w[1:].sum()
    plain = simulate(w, healthy)
    spec = simulate(w, healthy, speculate=True)
    print(f'{n:>8,}{plain["makespan"]:>9.1f} s{spec["makespan"]:>12.1f} s'
          f'{1 - spec["makespan"] / plain["makespan"]:>9.0%}{spec["clones"]:>9}'
          f'{spec["extra"]:>19,.0f} s{spec["extra"] / TOTAL_WORK_S:>14.1%}')
print()
print(f'the skewed task is {3 * FLOOR:.0f} s of work on an idle, healthy slot. '
      f'a second copy of it is also {3 * FLOOR:.0f} s.')

Speculation earns its keep where a single slow node defines the stage, which is the low task count
configuration, and its return shrinks as the task count rises and the scheduler starts balancing
around the slow nodes by itself. Note that it is not free in either case: the clone count barely
changes across the table, but at two hundred tasks each clone is a large task, so the duplicates add
roughly a fifth of the stage's real work in cluster seconds, against a couple of percent at four
thousand tasks. Wall clock improves, the bill goes up, and whether that trade is good depends on
which of the two you are being judged on.

The second block is the failure mode engines cannot detect. On a cluster with no slow nodes at all,
one partition carrying three floors' worth of work sets the makespan, and the clones change nothing
while still charging for themselves: a copy of a task reads the same partition and does the same
work. Speculation fixes slow machines. Skew is a partitioning problem, and a stage with one
permanently late task wants salting or a different partition key, not a second copy of the same long
task.

## Slots, and the limit of buying more

In [ ]:
w_fixed = DRAWS[1_000]
REPEATS = 5

print(f'{"slots":>8}{"slow":>7}{"makespan":>11}{"speedup":>10}'
      f'{"utilisation":>13}{"cluster seconds":>18}')
base = None
for slots in [25, 50, 100, 200, 400, 1_000, 2_000]:
    n_slow = max(1, int(round(slots * SLOW_SLOTS / SLOTS)))
    runs = []
    for _ in range(REPEATS):                      # average out where the slow slots land
        sp = np.ones(slots)
        sp[rng.choice(slots, n_slow, replace=False)] = SLOW_SPEED
        runs.append(simulate(w_fixed, sp))
    makespan = float(np.mean([r['makespan'] for r in runs]))
    util = float(np.mean([r['utilisation'] for r in runs]))
    base = base or makespan
    print(f'{slots:>8}{n_slow:>7}{makespan:>9.1f} s{base / makespan:>9.1f}x'
          f'{util:>13.1%}{makespan * slots:>16,.0f} s')
print()
print(f'longest task on a healthy slot {w_fixed.max() + LAUNCH_S:>8.1f} s')
print(f'the same task on a slow slot   {w_fixed.max() / SLOW_SPEED + LAUNCH_S:>8.1f} s')
print('past a few tasks per slot the scheduler runs out of work to balance with,')
print('so the makespan climbs back towards that second number.')

The speedup column peaks and then goes backwards, which is the result worth sitting with. Up to a few
tasks per slot, extra slots help. Past that, each slot holds about one task, the greedy loop has
nothing left to hand elsewhere, and the stage ends when one unlucky task finishes on a quarter speed
slot: the two numbers underneath the table are that task on a healthy slot and on a slow one, and the
makespan converges on the second. The last column is the other half of the bill. Cluster seconds rise
at every step, so the final doubling bought negative wall clock at twice the price.

That gives the order of operations for a slow stage. Look at the longest task first and decide
whether it is skew, because nothing else matters until that is answered. Then compare the task count
to the slot count, which is free to change. Only then add slots, and when you do, measure cluster
seconds as well as wall clock, because those two numbers move in opposite directions.

## What to take away

- A stage ends when its slowest task ends, so one task per slot makes the stage as slow as your worst
  node. Queued work is what lets a greedy scheduler route around a slow node at all.
- Task count has a U curve: slow slots and variance dominate the left arm, per task launch cost the
  right. The minimum is usually well past the two to four tasks per slot rule of thumb.
- Every barrier charges for the slowest task in the stage while the cluster idles. A small partition
  count late in a plan is the most common instance of this and the easiest to fix.
- Speculative execution helps at low task counts, costs little, and cannot fix skew: the clone reads
  the same oversized partition.
- More slots stop helping once the longest task dominates, and cluster seconds keep rising while wall
  clock does not. Fix the longest task first.